## Tardis Model

### 1. Package Importation

In [ ]:
import pandas as pd
import numpy as np
import joblib
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


In [ ]:
df = pd.read_csv("cleaned_dataset.csv")
cleaned_df = df

In [ ]:
df.info()

In [ ]:
df.isnull().sum() #allows you to see the number of null elements in the dataframe

### 1.1 Analysis on a global basis

In [ ]:
#convert string into values usable by the sklearn model
df = pd.get_dummies(df, columns=["Gare de départ", "Gare d'arrivée", "Service","Commentaire annulations","Commentaire retards à l'arrivée","Commentaire retards au départ"], drop_first=True)


#conversion of date "2022-08" to year 2022 and month 08
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m")
df["Annee"] = df["Date"].dt.year
df["Mois"] = df["Date"].dt.month
df = df.drop("Date", axis=1)


X = df.drop("Retard moyen de tous les trains à l'arrivée",axis=1) #definition of independent values
y = df["Retard moyen de tous les trains à l'arrivée"] #definition of target value

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
pipe = make_pipeline(StandardScaler(),LinearRegression()) 
pipe.fit(X_train, y_train) #model training


predictions = pipe.predict(X_test)  #first prediction

#comparison between real and predict value
print("MAE:", mean_absolute_error(y_test, predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, predictions)))
print("R² Score:", r2_score(y_test, predictions))    

In [ ]:
#export model
joblib.dump( 
    { 
        "model": pipe, 
        "features": list(X.columns),
        "X_columns": list(X.columns),
    },
    "model.joblib",
)

### 1.2 Preparation for real usage

In [ ]:
# creation dataframe with only four value which will be input in the dashboard
df_test = pd.DataFrame({
    "Gare de départ": ["LE MANS"],
    "Gare d'arrivée": ["PARIS MONTPARNASSE"],
    "Service": ["National"],
    "Date": ["2027-08"],
    "Durée moyenne du trajet": [None],
    "Nombre de circulations prévues": [None],
    "Nombre de trains annulés": [None],
    "Commentaire annulations": [None],
    "Nombre de trains en retard au départ": [None],
    "Retard moyen des trains en retard au départ": [None],
    "Retard moyen de tous les trains au départ": [None],
    "Commentaire retards au départ": [None],
    "Nombre de trains en retard à l'arrivée": [None],
    "Retard moyen des trains en retard à l'arrivée": [None],
    "Commentaire retards à l'arrivée": [None],
    "Nombre trains en retard > 15min": [None],
    "Retard moyen trains en retard > 15 (si liaison concurrencée par vol)": [None],
    "Nombre trains en retard > 30min": [None],
    "Nombre trains en retard > 60min": [None],
    "Prct retard pour causes externes": [None],
    "Prct retard pour cause infrastructure": [None],
    "Prct retard pour cause gestion trafic": [None],
    "Prct retard pour cause matériel roulant": [None],
    "Prct retard pour cause gestion en gare et réutilisation de matériel": [None],
    "Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)": [None]
})


def search_similate(cdf,df):

    #function whose purpose is to fill in a df from a cleaned df

    # retrive all data input
    GD = df["Gare de départ"].iloc[0]
    GA = df["Gare d'arrivée"].iloc[0]
    S = df["Service"].iloc[0]
    Date = df["Date"].iloc[0]
    
    #transfrom date in year and month
    target_date = datetime.strptime(Date, "%Y-%m")
    year = target_date.year
    month = target_date.month

    df["Annee"] = year
    df["Mois"] = month
    
    cdf["Date"] = pd.to_datetime(cdf["Date"], format="%Y-%m")
    
    cdf["Annee"] = cdf["Date"].dt.year
    cdf["Mois"] = cdf["Date"].dt.month
    
    # list of filters used to find the most similate row in the order of the most important filter
    filters = [
        (cdf["Gare d'arrivée"] == GA),
        (cdf["Mois"] == month),
        (cdf["Gare de départ"] == GD),
        (cdf["Annee"] == year),
        (cdf["Service"] == S)
    ]

    #loop that starts at the size of the list and decreases by 1 at a time
    for i in range(len(filters), 0, -1):
        #filter[:i] is a new list that corresponds to all values before index i.

        #mask retrive the first value of the new list
        mask = filters[:i][0]
        for filter in filters[:i][1:]:
            #loop for add the other filters with &
            mask = mask & filter        
        result = cdf[mask]  # retrieve corresponding line in the cleaned df
        print(result)
        if not result.empty: #if the result not []
            for col in df.columns:
                if pd.isna(df[col].iloc[0]) and col in result.columns:
                    # if the value is None in the df, and the col is in result,
                    # filling the df with the values from the cleaned df
                    df.at[0, col] = result.iloc[0][col]
            return True
    
    print("No matches found")
    return False


search_similate(
    cleaned_df,
    df_test
)



df_test = pd.get_dummies(
    df_test,
    columns=[
        "Gare de départ",
        "Gare d'arrivée",
        "Service",
        "Commentaire annulations",
        "Commentaire retards à l'arrivée",
        "Commentaire retards au départ"
    ],
    drop_first=True
)

df_test = df_test.drop("Date", axis=1)

#allows you to have the same column order as the df used for training
df_test = df_test.reindex(columns=X.columns, fill_value=0)

predictions = pipe.predict(df_test) #New prediction

print( "prediction : ",predictions[0]) #get the first prediction
